# Notebook 1: The Hydrogen Molecule — Building Intuition for Multireference Methods

**CSC Spring School on Computational Chemistry 2026**  
**Multireference Quantum Chemistry**

---

## Where we are coming from

In the earlier exercises you optimized molecular structures and calculated vibrational spectra with DFT. You gave ORCA a geometry, a functional, and a basis set, and it handled the rest. The electrons were doing their job quietly in the background.

If you come from a molecular dynamics or force field background, the electrons are even more invisible — baked into parameters, never explicitly treated. A bond is a spring. An atom is a sphere with a charge. It works remarkably well for equilibrium structures and dynamics near the minimum.

This session makes the electrons visible. And when you look closely, they turn out to be quite different from classical particles.

Two facts about electrons in quantum mechanics are worth keeping in mind throughout:

- **Electrons are correlated.** Every electron in a molecule is simultaneously aware of every other electron. The quantum mechanical description captures this through a single joint object — the wavefunction — that encodes all electrons together. This is not a technical detail. It is the origin of chemistry.

- **Electrons are indistinguishable.** There is no electron number 1 and electron number 2. There is only "two electrons", and the mathematics must reflect that. This has deep consequences for how we construct wavefunctions.

For most molecules near their equilibrium geometry, good approximate methods exist that handle both facts well enough. But when a bond breaks, the electronic structure changes in a way that exposes the limits of the most common approximations. The hydrogen molecule — two protons, two electrons, one bond — is the cleanest possible example of this.

---

## Dual motivation

*If you work on biological systems:* The concepts developed here — static correlation, natural orbital occupations, the failure of a single determinant — are directly relevant to enzyme active sites, metal cofactors, and photochemical processes in proteins and DNA. The hydrogen molecule is the transparent model that builds the intuition you need for those systems.

*If you work on atmospheric or physical chemistry:* Bond breaking is central to radical chemistry, combustion, and atmospheric reactions. The failure modes of single-reference methods that we demonstrate here for H₂ are the same ones that make O₂, O₃, and NOₓ chemistry challenging.

---

## Learning objectives

By the end of this notebook you will be able to:

- Explain how atomic orbitals combine to form molecular orbitals, and why there are exactly two combinations
- Describe the single determinant ansatz and its physical meaning
- Identify where and why RHF and DFT fail for bond breaking
- Understand spin contamination in UHF as a symptom of the underlying problem
- Interpret natural orbital occupation numbers as a diagnostic for multireference character
- Explain qualitatively why CASSCF(2,2) gives the correct dissociation limit

---

## Setup

In [ ]:
import sys
sys.path.insert(0, '../tools')

import os
import re
import subprocess
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
from IPython.display import Image

from utils import (
    ORCA, NPROCS, HARTREE_TO_KJMOL,
    setup_workdir, clean_workdir, _count_stale,
    run_orca, get_energy, get_s2, get_no_occupations,
    terminated_normally,
)

print(f'ORCA:   {ORCA}')
print(f'NPROCS: {NPROCS}')

In [ ]:
import shutil

WORKDIR     = 'h2'
FORCE_CLEAN = False

if FORCE_CLEAN:
    if Path(WORKDIR).exists():
        shutil.rmtree(WORKDIR)
        print(f'✓  Removed {WORKDIR}/ — all previous results deleted.')

work_dir = setup_workdir(WORKDIR)

if not FORCE_CLEAN:
    stale = _count_stale(work_dir)
    if stale:
        print(f'⚠️  {work_dir}/ contains {len(stale)} files from a previous run.')
        print( '   Set FORCE_CLEAN = True to remove everything and start fresh.')

---
## Part 0: Building the picture — from two atoms to a molecule

Before running any calculations, we build the conceptual picture step by step. All figures in this section are generated from the analytical hydrogen wavefunction — no ORCA needed yet.

### Two non-interacting hydrogen atoms

Start from what we know: two separate hydrogen atoms, far apart. Each atom has one proton and one electron. The electron of atom A is described by the 1s wavefunction $\phi_{1s}^A$, and similarly for atom B:

$$\phi_{1s}(r) = K \, e^{-r/a_0}$$

where $a_0 = 0.529$ Å is the Bohr radius and K is a normalization constant. This is not a physical orbit — it is a mathematical function whose square gives the **probability density** of finding the electron at position r.

The figure below shows this probability density in a 2D slice through the atom.

> **Orbitals are mathematical objects, not physical trajectories.** The electron does not travel around the nucleus. The orbital describes our knowledge of where the electron *might be found* if we measured its position.

In [ ]:
a0 = 0.529  # Bohr radius in Angstrom

def psi_1s(x, y, x0=0.0, y0=0.0):
    """Hydrogen 1s wavefunction in the z=0 plane."""
    r = np.sqrt((x - x0)**2 + (y - y0)**2)
    return (1.0 / (np.sqrt(np.pi) * a0**1.5)) * np.exp(-r / a0)

x = np.linspace(-4, 4, 400)
y = np.linspace(-3, 3, 400)
X, Y = np.meshgrid(x, y)

fig, ax = plt.subplots(figsize=(4, 3.5))
prob = psi_1s(X, Y)**2
ax.contourf(X, Y, prob, levels=20, cmap='Blues')
ax.contour(X, Y, prob, levels=8, colors='navy', linewidths=0.5, alpha=0.5)
ax.plot(0, 0, 'o', color='red', ms=7, zorder=5, label='proton')
ax.set_xlabel('x (Å)', fontsize=11)
ax.set_ylabel('y (Å)', fontsize=11)
ax.set_title('H atom: 1s probability density |ψ|²', fontsize=11)
ax.set_aspect('equal')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### The atoms approach — building the molecular wavefunction

Now bring the two atoms together. When they are far apart and non-interacting, the simplest description of the two-electron system is just the product of the two atomic wavefunctions:

$$\Psi^{(0)} = \phi_{1s}^A(1) \cdot \phi_{1s}^B(2)$$

This says: electron 1 belongs to atom A, electron 2 belongs to atom B. But this is already wrong — it assigns labels to the electrons, treating them as distinguishable particles. Electrons are **indistinguishable**: there is no electron 1 and electron 2, only 'two electrons'.

The antisymmetry principle — a fundamental postulate of quantum mechanics for fermions, whose experimental basis is the periodic table itself — requires that the wavefunction changes sign when any two electrons are exchanged:

$$\Psi(1,2) = -\Psi(2,1)$$

The simple product does not satisfy this. We must antisymmetrize it. Including spin, the correctly antisymmetrized wavefunction for the singlet ground state is:

$$\Psi_{HL} = [\phi_{1s}^A(1)\phi_{1s}^B(2) + \phi_{1s}^B(1)\phi_{1s}^A(2)]\cdot[\alpha(1)\beta(2) - \beta(1)\alpha(2)]$$

This is the **Heitler-London wavefunction** — the original valence bond description of H₂, published in 1927. The spatial part is symmetric (both electrons visit both atoms equally), and the spin part is antisymmetric (one spin up, one spin down, but indistinguishably). Their product is antisymmetric overall.

This wavefunction is **purely covalent** by construction — there are no terms where both electrons are on the same atom. This is exactly right at dissociation: each atom gets one electron.

The figure below shows how the electron density of the bonding combination changes as the atoms approach.

> **Why does the periodic table imply antisymmetry?** If electrons were not subject to the Pauli exclusion principle — which follows directly from antisymmetry — all electrons in an atom would collapse into the 1s orbital. There would be no shell structure, no periodicity, no chemistry. The fact that lithium must put its third electron in 2s, that the transition metals have partially filled d shells, that helium is inert — all of this is direct experimental evidence for the antisymmetry of the wavefunction.

In [ ]:
r_seq = [4.0, 2.0, 1.2, 0.74]
fig, axes = plt.subplots(1, 4, figsize=(14, 3))

for ax, r in zip(axes, r_seq):
    psi_A = psi_1s(X, Y, x0=-r/2)
    psi_B = psi_1s(X, Y, x0=+r/2)
    prob  = (psi_A + psi_B)**2
    ax.contourf(X, Y, prob, levels=15, cmap='Blues')
    ax.plot(-r/2, 0, 'o', color='red', ms=5, zorder=5)
    ax.plot(+r/2, 0, 'o', color='red', ms=5, zorder=5)
    ax.set_title(f'r = {r} Å', fontsize=10)
    ax.set_aspect('equal')
    ax.set_xlim(-3, 3)
    ax.set_ylim(-2, 2)
    ax.set_xlabel('x (Å)', fontsize=8)
    if r == r_seq[0]:
        ax.set_ylabel('y (Å)', fontsize=8)
    else:
        ax.set_yticklabels([])

plt.suptitle('σ orbital electron density as H atoms approach', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
r_values = np.linspace(0.3, 5.0, 200)
x1d = np.linspace(-8, 8, 2000)
overlaps = []
for r in r_values:
    pA = (1.0/(np.sqrt(np.pi)*a0**1.5)) * np.exp(-np.abs(x1d + r/2)/a0)
    pB = (1.0/(np.sqrt(np.pi)*a0**1.5)) * np.exp(-np.abs(x1d - r/2)/a0)
    overlaps.append(np.trapezoid(pA * pB, x1d))

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(r_values, overlaps, 'b-', lw=2.5)
ax.axvline(0.74, color='red', ls='--', lw=1.5, label='r$_{eq}$ = 0.74 Å')
ax.axhline(0, color='grey', lw=0.8, ls=':')
ax.set_xlabel('H–H distance (Å)', fontsize=11)
ax.set_ylabel('Overlap integral S', fontsize=11)
ax.set_title('1s–1s orbital overlap as H atoms approach', fontsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

### Adding the interaction correction — and the LCAO approximation

The Heitler-London wavefunction is a good starting point, but it is not an eigenfunction of the molecular Hamiltonian. As the atoms approach, the electrons feel the attraction of *both* nuclei — they occasionally visit the other atom. We need to make the wavefunction more flexible by allowing for this.

The simplest correction adds **ionic terms** — configurations where both electrons are on the same atom:

$$\Psi = \underbrace{[\phi_A(1)\phi_B(2) + \phi_B(1)\phi_A(2)]}_{\text{covalent}} + \lambda\underbrace{[\phi_A(1)\phi_A(2) + \phi_B(1)\phi_B(2)]}_{\text{ionic}}$$

where $\lambda$ is a variational parameter that the calculation optimizes. Near equilibrium $\lambda$ is small but nonzero — the electrons do occasionally visit the same nucleus. At dissociation $\lambda \to 0$ — the purely covalent Heitler-London limit is recovered, with each atom getting one electron.

**This is the physically correct picture.** Now — how does the standard MO approach relate to this?

### The MO approach — LCAO

Instead of starting from atomic wavefunctions and adding corrections, the molecular orbital approach builds new one-electron functions — **molecular orbitals** — that extend over the whole molecule. We choose to build these as linear combinations of atomic orbitals (LCAO). This is a practical and intuitive choice, not a theoretical necessity — any complete set of functions would work, but atomic orbitals give a natural starting point and chemical interpretability.

For H₂, symmetry dictates the only two valid combinations:

$$\sigma_g = K_+\,(\phi_{1s}^A + \phi_{1s}^B) \qquad \text{(gerade — symmetric)}$$

$$\sigma_u^* = K_-\,(\phi_{1s}^A - \phi_{1s}^B) \qquad \text{(ungerade — antisymmetric)}$$

Any other combination would break the inversion symmetry of the molecule. The symmetric combination $\sigma_g$ has electron density concentrated *between* the nuclei — a bonding orbital. The antisymmetric combination $\sigma_u^*$ has a node between the nuclei — an antibonding orbital.

The figure below shows both molecular orbitals at the equilibrium geometry.

In [ ]:
r_eq = 0.74
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

for ax, sign, title, cmap in zip(
        axes,
        [+1, -1],
        ['σ bonding: ψ_A + ψ_B\n(constructive interference — density between nuclei)',
         'σ* antibonding: ψ_A − ψ_B\n(destructive interference — node between nuclei)'],
        ['Blues', 'Reds']):
    psi_A = psi_1s(X, Y, x0=-r_eq/2)
    psi_B = psi_1s(X, Y, x0=+r_eq/2)
    mo    = psi_A + sign * psi_B
    prob  = mo**2
    ax.contourf(X, Y, prob, levels=20, cmap=cmap)
    ax.contour(X, Y, prob, levels=8, colors='black', linewidths=0.4, alpha=0.3)
    if sign == -1:
        ax.axvline(0, color='white', lw=2, ls='--', label='node')
        ax.legend(fontsize=9)
    ax.plot(-r_eq/2, 0, 'o', color='red', ms=6, zorder=5)
    ax.plot(+r_eq/2, 0, 'o', color='red', ms=6, zorder=5)
    ax.set_xlabel('x (Å)', fontsize=10)
    ax.set_ylabel('y (Å)', fontsize=10)
    ax.set_title(title, fontsize=10)
    ax.set_aspect('equal')
    ax.set_xlim(-3, 3)
    ax.set_ylim(-2, 2)

plt.tight_layout()
plt.show()

### The energy level diagram

As the atoms approach, the two 1s atomic orbital energies split into two molecular orbital energies. The bonding orbital goes down in energy — stabilization — while the antibonding orbital goes up. The figure below shows this schematically as a function of the H–H distance.

In [ ]:
r_vals = np.linspace(0.5, 5.0, 200)
x1d    = np.linspace(-8, 8, 2000)
S_vals = np.array([
    np.trapezoid(
        (1.0/(np.sqrt(np.pi)*a0**1.5)) * np.exp(-np.abs(x1d+r/2)/a0) *
        (1.0/(np.sqrt(np.pi)*a0**1.5)) * np.exp(-np.abs(x1d-r/2)/a0),
        x1d)
    for r in r_vals])

E_1s   = -13.6   # eV
beta   = -30.0   # eV (schematic resonance integral)
E_sig  = E_1s + beta * S_vals
E_sigs = E_1s - beta * S_vals

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(r_vals, E_sig,  'b-',  lw=2.5, label='σ bonding orbital')
ax.plot(r_vals, E_sigs, 'r--', lw=2.5, label='σ* antibonding orbital')
ax.axhline(E_1s, color='grey', lw=1.2, ls=':', label='H 1s energy')
ax.axvline(0.74, color='green', lw=1.5, ls='--', alpha=0.8, label='r$_{eq}$ = 0.74 Å')
ax.annotate('both electrons\nin σ orbital', xy=(0.74, E_sig[np.argmin(np.abs(r_vals-0.74))]),
            xytext=(1.5, -35), fontsize=9,
            arrowprops=dict(arrowstyle='->', color='green'), color='green')
ax.set_xlabel('H–H distance (Å)', fontsize=11)
ax.set_ylabel('Orbital energy (eV, schematic)', fontsize=11)
ax.set_title('Molecular orbital energies as H atoms approach', fontsize=11)
ax.set_xlim(0.5, 5.0)
ax.set_ylim(-55, 20)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

### The single determinant — and where it breaks down

The Hartree-Fock wavefunction places both electrons in $\sigma_g$ with opposite spins, written as a Slater determinant:

$$\Psi_{HF} = \sigma_g^2\,[\alpha(1)\beta(2) - \beta(1)\alpha(2)]$$

Now expand $\sigma_g^2$ in atomic orbitals:

$$\sigma_g(1)\sigma_g(2) = K_+^2(\phi_A+\phi_B)(1)(\phi_A+\phi_B)(2)$$

$$= K_+^2[\underbrace{\phi_A(1)\phi_B(2)+\phi_B(1)\phi_A(2)}_{\text{covalent}} + \underbrace{\phi_A(1)\phi_A(2)+\phi_B(1)\phi_B(2)}_{\text{ionic}}]$$

Comparing with the valence bond picture:

$$\Psi = \text{covalent} + \lambda \cdot \text{ionic}$$

RHF corresponds to **$\lambda = 1$ at all distances** — equal weights for covalent and ionic terms, always. Near equilibrium this is a reasonable approximation — the ionic contribution is small and the error is manageable. But at dissociation $\lambda$ should approach zero — the wavefunction should become purely covalent. RHF cannot do this. It is stuck at $\lambda = 1$, predicting 50% H⁺···H⁻ character at infinite separation. This is unphysical and gives a completely wrong dissociation energy.

The dissociation limit provides a hard consistency check: at $r \to \infty$ the two hydrogen atoms are non-interacting and the exact energy must be:

$$E(r\to\infty) = 2 \times E_H = 2 \times (-13.6\text{ eV}) = -27.2\text{ eV}$$

Any method that gets the wrong dissociation limit has a **qualitative failure** — not a quantitative error.

**How CASSCF fixes this:**

CASSCF uses a two-determinant wavefunction:

$$\Psi = C_1|\sigma_g\bar\sigma_g\rangle + C_2|\sigma_u^*\bar\sigma_u^*\rangle$$

Expanding $|\sigma_u^*\bar\sigma_u^*\rangle$ in AOs:

$$|\sigma_u^*\bar\sigma_u^*\rangle = K_-^2[\underbrace{\phi_A(1)\phi_B(2)+\phi_B(1)\phi_A(2)}_{\text{covalent}} - \underbrace{\phi_A(1)\phi_A(2)+\phi_B(1)\phi_B(2)}_{\text{ionic}}]$$

The ionic terms appear with a **minus sign** — so in the combination $C_1|\sigma_g\bar\sigma_g\rangle + C_2|\sigma_u^*\bar\sigma_u^*\rangle$ the ionic contribution is:

$$(C_1 K_+^2 - C_2 K_-^2)[\phi_A\phi_A + \phi_B\phi_B]$$

At dissociation ($S\to 0$, $K_+ = K_- = 1/\sqrt{2}$) this vanishes when $C_1 = C_2$ — which is exactly what CASSCF finds variationally. The result is the Heitler-London wavefunction: purely covalent, $\lambda = 0$, correct dissociation energy.

CASSCF is not doing something exotic — it is recovering what Heitler and London wrote down in 1927, expressed in the MO basis.

> **Why only double excitations help:** The ionic-cancelling term $|\sigma_u^*\bar\sigma_u^*\rangle$ is a *double* excitation from the HF reference. Single excitations (CIS) cannot access it — which is why CIS cannot fix the dissociation problem regardless of how many singly excited determinants are included.

> **The connection to NO occupations:** The coefficient $C_2$ grows as the bond stretches. When $C_2 \approx C_1$ the system has strong multireference character — and this is exactly what the natural orbital occupation numbers measure. When the $\sigma_u^*$ occupation approaches 1.0, $C_2$ is approaching $C_1$.

---
## Part 1: A single RHF calculation at equilibrium

Let's start with one RHF calculation at the equilibrium geometry (r = 0.74 Å) to get familiar with the ORCA input format before running the full scan.

Key parts of the input:
- `! RHF def2-SVP TightSCF` — method and basis set on the simple input line
- `%pal nprocs N end` — number of CPU cores
- `* xyz charge multiplicity ... *` — molecular geometry

> **Why def2-SVP?** Small but balanced — fast enough for a 16-point × 4-method scan, large enough to show the physics clearly.

In [ ]:
inp = work_dir / 'h2_rhf.inp'
inp.write_text("""! RHF def2-SVP TightSCF
%pal nprocs 1 end
* xyz 0 1
H  0.0000  0.0000  0.0000
H  0.7400  0.0000  0.0000
*
""")
print("Input written to", inp)

> ⏳ Run the cell below and wait for it to finish before continuing.

In [ ]:
out = run_orca('h2_rhf', """! RHF def2-SVP TightSCF
%pal nprocs 1 end
* xyz 0 1
H  0.0000  0.0000  0.0000
H  0.7400  0.0000  0.0000
*
""", work_dir=work_dir)

print(f"Terminated normally: {terminated_normally(out)}")
print(f"RHF energy at r=0.74 Å: {get_energy(out):.6f} Eh")

You can open `h2/h2_rhf.out` in the file browser on the left to explore the full ORCA output.

**Q0.** The total RHF energy is negative and in units of Hartree (1 Eh = 27.21 eV = 627.5 kcal/mol). Does the sign make sense? What does a negative total energy mean physically?

*Your answer:*

---
## Part 2: The potential energy curve — three methods compared

Now we scan the H–H distance from 0.50 Å to 5.0 Å and compute the energy with three methods:

| Method | Description |
|--------|-------------|
| RHF | Restricted HF — single determinant, both electrons forced into σ_g |
| B3LYP | Hybrid DFT — single determinant with exchange-correlation functional |
| CASSCF(2,2) | 2 electrons in 2 orbitals (σ_g, σ_u*) — minimal multireference method |

The scan includes points on the repulsive wall (r < 0.74 Å) to show that the equilibrium geometry is a true minimum, and extends to r = 5.0 Å to show the dissociation behaviour.

> ⏳ **Please be patient.** This scan runs 19 geometries × 3 methods. Wait for the `[*]` indicator to change to a number before continuing.

In [ ]:
r_values = np.array([0.50, 0.55, 0.60, 0.65, 0.70, 0.74, 0.80,
                     0.90, 1.00, 1.10, 1.20, 1.40, 1.60, 1.80,
                     2.00, 2.50, 3.00, 4.00, 5.00])

# ── Step 1: hidden CASSCF at equilibrium to seed orbitals ────────────
_geom_eq  = 'H  0.0000  0.0000  0.0000\nH  0.7400  0.0000  0.0000'
_seed_out = work_dir / 'h2_seed_casscf.out'
_seed_gbw = 'h2_seed_casscf.gbw'

if not _seed_out.exists():
    run_orca('h2_seed_casscf',
             '! CASSCF def2-SVP TightSCF\n'
             '%pal nprocs 1 end\n'
             '%casscf\n'
             '  nel    2\n'
             '  norb   2\n'
             '  nroots 1\n'
             '  ActConstraints 1\n'
             '  printlevel 3\n'
             'end\n'
             f'* xyz 0 1\n{_geom_eq}\n*\n',
             work_dir=work_dir)
    print('Seed CASSCF: ' +
          ('OK' if terminated_normally(_seed_out) else 'FAILED'))
else:
    print('Seed CASSCF: already exists')

# ── Step 2: full scan ─────────────────────────────────────────────────
# Short r: use seed orbitals (stable at compressed geometries)
# Long r:  chain from previous converged point

energies = {k: [] for k in ['RHF', 'B3LYP', 'CASSCF']}
prev_gbw = _seed_gbw

for r in r_values:
    tag  = f"h2_{r:.2f}".replace('.', 'p')
    geom = f"H  0.0000  0.0000  0.0000\nH  {r:.4f}  0.0000  0.0000"

    inp_rhf = (
        '! RHF def2-SVP TightSCF\n'
        '%pal nprocs 1 end\n'
        f'* xyz 0 1\n{geom}\n*\n'
    )
    inp_b3lyp = (
        '! B3LYP def2-SVP TightSCF\n'
        '%pal nprocs 1 end\n'
        f'* xyz 0 1\n{geom}\n*\n'
    )
    inp_casscf = (
        '! CASSCF def2-SVP TightSCF MOREAD\n'
        '%pal nprocs 1 end\n'
        f'%moinp "{prev_gbw}"\n'
        '%casscf\n'
        '  nel    2\n'
        '  norb   2\n'
        '  nroots 1\n'
        '  ActConstraints 1\n'
        '  printlevel 3\n'
        'end\n'
        f'* xyz 0 1\n{geom}\n*\n'
    )

    for method, inp in [('RHF', inp_rhf),
                         ('B3LYP', inp_b3lyp),
                         ('CASSCF', inp_casscf)]:
        out_file = work_dir / f"{tag}_{method.lower()}.out"
        if not out_file.exists():
            run_orca(f"{tag}_{method.lower()}", inp, work_dir=work_dir)
        energies[method].append(get_energy(out_file))

    # Chain MOREAD if CASSCF converged
    cas_out = work_dir / f"{tag}_casscf.out"
    if terminated_normally(cas_out):
        prev_gbw = f"{tag}_casscf.gbw"

    cas_ok = terminated_normally(cas_out)
    print(f"r = {r:.2f} Å  CASSCF: {'OK' if cas_ok else 'FAILED'}  "
          f"E = {energies['CASSCF'][-1]:.5f} Eh", flush=True)

for k in energies:
    energies[k] = np.array(energies[k])

print('\n✓ Scan complete.')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

colors  = {'RHF': '#e74c3c', 'B3LYP': '#8e44ad', 'CASSCF': '#27ae60'}
styles  = {'RHF': '-',       'B3LYP': '-.',       'CASSCF': '-'}
markers = {'RHF': 'o',       'B3LYP': '^',        'CASSCF': 'D'}

for method, E in energies.items():
    E_ref = np.nanmin(energies['CASSCF'])
    E_rel = (E - E_ref) * HARTREE_TO_KJMOL
    mask  = ~np.isnan(E_rel)
    ax.plot(r_values[mask], E_rel[mask],
            color=colors[method], ls=styles[method],
            marker=markers[method], ms=5, label=method, lw=2)

ax.axvline(0.74, color='grey', lw=1, ls=':', alpha=0.7,
           label='r$_{eq}$ = 0.74 Å')
ax.axhline(0, color='grey', lw=0.5, ls=':')
ax.set_xlabel('H–H distance (Å)', fontsize=13)
ax.set_ylabel('Relative energy (kJ/mol)', fontsize=13)
ax.set_title('H₂ Potential Energy Curves — def2-SVP', fontsize=14)
ax.set_xlim(0.4, 5.2)
ax.set_ylim(-80, 700)
ax.legend(fontsize=12)
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
plt.tight_layout()
plt.savefig(work_dir / 'h2_pec.pdf')
plt.show()

### ✏️ Questions — Part 2

**Q1.** At large H–H distance (r > 3 Å) the RHF and B3LYP curves rise steeply instead of reaching a plateau. Based on the valence bond picture we developed, why does this happen? What unphysical ionic contribution does the single determinant retain at large r?

*Your answer:*

---

**Q2.** At what H–H distance do the RHF and B3LYP curves start to deviate significantly from CASSCF? Is this consistent with where you would expect the ionic contribution to become important?

*Your answer:*

---

**Q3.** The experimental dissociation energy of H₂ is 436 kJ/mol. Which method comes closest? Read the value from the plot as the energy difference between the minimum and the plateau at large r.

*Your answer:*

---

**Q4.** B3LYP is a DFT method that goes beyond HF by including an exchange-correlation functional. Yet it fails at dissociation just like RHF. Why? What does this tell you about the fundamental limitation that DFT shares with RHF?

*Your answer:*

---

**Q5.** CASSCF gives the correct dissociation limit. Looking at the PEC, does CASSCF also give the correct equilibrium bond length and dissociation energy? What might be missing from CASSCF that would improve these?

*Your answer:*

---
## Part 3: Natural orbital occupation numbers

The CASSCF calculation gives us access to **natural orbitals (NOs)** and their **occupation numbers**. These are the eigenfunctions of the one-particle density matrix — the orbitals that most compactly describe the electron distribution.

In a single-determinant wavefunction, occupation numbers are always exactly 0 or 2 (empty or doubly occupied). In a multireference wavefunction, they take non-integer values between 0 and 2.

For CASSCF(2,2) on H₂:
- At equilibrium: σ occupation ≈ 2.0, σ* occupation ≈ 0.0 → single reference
- At dissociation: σ occupation ≈ 1.0, σ* occupation ≈ 1.0 → strong multireference character

The deviation of the occupation numbers from 0 and 2 is a direct, quantitative measure of the multireference character of the wavefunction.

In [ ]:
no_bonding     = []
no_antibonding = []

for r in r_values:
    tag  = f"h2_{r:.2f}".replace('.', 'p')
    occs = get_no_occupations(work_dir / f"{tag}_casscf.out")
    no_bonding.append(occs[0]    if len(occs) > 0 else float('nan'))
    no_antibonding.append(occs[1] if len(occs) > 1 else float('nan'))

no_bonding     = np.array(no_bonding)
no_antibonding = np.array(no_antibonding)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(r_values, no_bonding,     'o-',  color='#2980b9', lw=2, ms=6,
        label='σ (bonding NO)')
ax.plot(r_values, no_antibonding, 's--', color='#c0392b', lw=2, ms=6,
        label='σ* (antibonding NO)')
ax.axhline(1.0, color='grey', lw=0.8, ls=':', label='occ = 1.0')
ax.set_xlabel('H–H distance (Å)', fontsize=12)
ax.set_ylabel('NO occupation number', fontsize=12)
ax.set_title('Natural orbital occupations — CASSCF(2,2)/def2-SVP', fontsize=12)
ax.set_ylim(-0.05, 2.05)
ax.legend(fontsize=11)
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
plt.tight_layout()
plt.savefig(work_dir / 'h2_no_occupations.pdf')
plt.show()

### Orbital gallery — σ_g and σ_u* at key geometries

The occupation numbers tell us *how much* each orbital contributes to the wavefunction. The cube files let us see *what* those orbitals look like. The cell below uses `orca_plot` to generate cube files from the CASSCF `.gbw` files and displays them with py3Dmol.

Blue = positive phase, Red = negative phase.

> **Note:** This uses `orca_plot` which ships with ORCA. The `.gbw` files must exist in the working directory — i.e. the scan must have been run first.

In [ ]:
from utils import plot_orbital, show_orbital, orbital_gallery

# Show σ_g (MO 0) and σ_u* (MO 1) at three geometries
for r, description in [(0.74, 'equilibrium'), 
                        (1.50, 'stretched'),
                        (3.00, 'dissociation')]:
    tag = f"h2_{r:.2f}".replace('.', 'p')
    gbw = work_dir / f"{tag}_casscf.gbw"

    if not gbw.exists():
        print(f"Skipping r={r} Å — {gbw.name} not found")
        continue

    # Get occupation numbers for annotation
    occs = get_no_occupations(work_dir / f"{tag}_casscf.out")
    occ_g  = occs[0] if len(occs) > 0 else None
    occ_u  = occs[1] if len(occs) > 1 else None

    print(f"\n{'='*50}")
    print(f"r = {r} Å  ({description})")
    print(f"{'='*50}")
    orbital_gallery(
        gbw,
        mo_indices=[0, 1],
        occupations=[occ_g, occ_u],
        labels=['σ_g (bonding)', 'σ_u* (antibonding)'],
        isoval=0.05,
        ngrid=40,
        work_dir=work_dir,
    )

### ✏️ Fill in the table

Read the values from the plot and complete the table:

| r (Å) | σ occupation | σ* occupation | Character |
|--------|-------------|----------------|-----------|
| 0.74 | | | single-reference / multireference |
| 1.50 | | | single-reference / multireference |
| 2.50 | | | single-reference / multireference |
| 5.00 | | | single-reference / multireference |

A practical rule of thumb: if the two frontier NO occupations deviate significantly from (2.0, 0.0) — say both between 1.8 and 0.2 — multireference treatment is needed.

---
### Orbital gallery — σ_g and σ_u* at key geometries

The occupation numbers tell us *how much* each orbital contributes to the wavefunction. Seeing the orbitals at three geometries — equilibrium, stretched, and near-dissociation — shows how the electronic structure changes as the bond breaks.

Blue = positive phase, red = negative phase.

> The `.gbw` files must exist — the scan must have been run first.

In [ ]:
from utils import orbital_gallery

# Show σ_g (MO 0) and σ_u* (MO 1) at three geometries
for r, description in [(0.74, 'equilibrium'),
                        (1.50, 'stretched'),
                        (3.00, 'dissociation')]:
    tag      = f"h2_{r:.2f}".replace('.', 'p')
    basename = f"{tag}_casscf"
    gbw      = work_dir / f"{basename}.gbw"

    if not gbw.exists():
        print(f"Skipping r={r} Å — {gbw.name} not found")
        continue

    occs  = get_no_occupations(work_dir / f"{basename}.out")
    occ_g = occs[0] if len(occs) > 0 else None
    occ_u = occs[1] if len(occs) > 1 else None

    print(f"\n{'='*50}")
    print(f"r = {r} Å  ({description})")
    print(f"{'='*50}")

    orbital_gallery(
        basename,
        mo_indices=[0, 1],
        occupations=[occ_g, occ_u],
        labels=['σ_g (bonding)', 'σ_u* (antibonding)'],
        isoval=0.08,
        resolution=60,
        work_dir=work_dir,
    )

### ✏️ Questions — Part 4

**Q7.** At what H–H distance do the σ and σ* occupation numbers cross 1.0? Compare this with the distance at which ⟨S²⟩ starts to deviate in Part 3. Are they consistent?

*Your answer:*

---

**Q8.** The NO occupation numbers are computed from the CASSCF wavefunction. Could you extract the same information from a DFT calculation? Why or why not?

*Your answer:*

---
## Part 4: A note on CCSD and the full CI limit

For H₂ with two electrons, CCSD is **exact** — it equals Full Configuration Interaction (FCI). Singles and doubles exhaust the full CI space for a two-electron system. This makes H₂ a clean theoretical baseline: CASSCF(2,2) and CCSD are both formally exact for H₂.

For larger molecules, CCSD is no longer exact. The **T1 diagnostic** — a measure of the weight of single excitations in the CCSD wavefunction — is a practical indicator of multireference character:

- T1 < 0.02: single-reference methods are reliable
- T1 > 0.02: multireference character is significant, results should be treated with caution

This diagnostic is routinely computed as a first check before deciding whether to use multireference methods for a new system.

---
## Summary

Fill in the blanks:

1. The Heitler-London wavefunction is **purely covalent** — it contains no **_____** terms. This is physically correct at dissociation because each atom must recover exactly **_____** electron.

2. The RHF wavefunction expanded in AOs contains equal weights of covalent and **_____** terms at all distances. At dissociation this corresponds to 50% **_____** character — unphysical.

3. CASSCF(2,2) adds the **_____** determinant $|\sigma_u^*\bar\sigma_u^*\rangle$ to the wavefunction. This term has ionic contributions with a **_____** sign, so when $C_1 = C_2$ the ionic terms cancel exactly.

4. At equilibrium the σ_g NO occupation ≈ **_____** and σ_u* ≈ **_____**. The wavefunction is predominantly **single-reference / multireference** (circle one).

5. At dissociation both NO occupations ≈ **_____**. This indicates **static / dynamic** (circle one) correlation.

6. The coefficient $C_2$ in the CASSCF expansion is directly related to the **_____** occupation number. When this approaches **_____**, the system has strong multireference character.

7. CCSD = FCI for H₂ because with only **_____** electrons, singles and doubles exhaust the full CI space.

---

*Continue to Notebook 2: N₂ — where we ask how hard it is to get the atomisation energy right.*